<a href="https://colab.research.google.com/github/JOYCIDA/HELLO-WORLD/blob/main/Assignment3_HuggingFace_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — LLM Invocation with HuggingFace & LangChain
**Course:** Introduction to Artificial Intelligence  
**Name:** MOUAHA JOYCE    

---
This notebook demonstrates four different approaches to loading and invoking large language models using HuggingFace Transformers and LangChain. Each task builds on the previous one, introducing a new pattern or concept. Tasks 1–3 use TinyLlama. Task 4 uses Qwen2-0.5B-Instruct with a photography assistant use case.


## Setup — Install Required Packages
The cell below installs all libraries needed for this assignment:
- **langchain** — core LangChain framework for chaining LLM calls
- **langchain-community** — older integration package (used in Task 3)
- **langchain-huggingface** — newer dedicated HuggingFace integration (used in Task 1 and 4)
- **transformers** — HuggingFace library for loading and running models
- **accelerate** — enables automatic CPU/GPU device mapping
- **torch** — PyTorch deep learning framework that runs the models


In [1]:
!pip install langchain langchain-community langchain-huggingface transformers accelerate torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


---
## Task 1 — Load TinyLlama with `pipeline()` + `HuggingFacePipeline` (langchain_huggingface)

In this task I load TinyLlama using the `pipeline()` convenience function from HuggingFace Transformers and wrap it with `HuggingFacePipeline` from the newer `langchain_huggingface` package. This is the simplest approach — it handles tokenization, inference, and decoding automatically behind one function call.

### What each `pipeline()` parameter controls

| Parameter | What it controls |
|-----------|-----------------|
| `"text-generation"` | The **task type** — tells HuggingFace this is a decoder-only text generation model, not a classifier or translator |
| `model` | The **HuggingFace Hub model ID** — the library downloads and caches the weights automatically |
| `max_new_tokens=200` | The **maximum tokens** the model can generate. One token ≈ 0.75 words. Prevents infinite generation. |
| `temperature=0.7` | Controls **randomness**. 0.0 = fully deterministic. 1.0 = very creative. 0.7 is a balanced default. |

### System persona
I chose a technical explainer persona for AI engineering students — distinct from the generic "helpful assistant" in the lab example.


In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Load TinyLlama via pipeline() — handles tokenization and decoding automatically
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    temperature=0.7
)

# Wrap in LangChain so we can call .invoke() — uses the newer recommended import path
llm = HuggingFacePipeline(pipeline=pipe)

# Chat template format with a custom persona and a specific AI question
# The system message is different from the lab example
prompt = """<|system|>
You are a precise and concise technical explainer specialised in AI and machine learning.
You always structure your answers with numbered points and avoid unnecessary filler phrases.
<|user|>
What are the three most important things an engineering student must understand about
how attention mechanisms work in transformer models?
<|assistant|>
"""

response = llm.invoke(prompt)
print(response)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>
You are a precise and concise technical explainer specialised in AI and machine learning.
You always structure your answers with numbered points and avoid unnecessary filler phrases.
<|user|>
What are the three most important things an engineering student must understand about
how attention mechanisms work in transformer models?
<|assistant|>
Here are the three most important things an engineering student must understand about attention mechanisms in transformer models:
1. Self-Attention: In transformer models, attention mechanisms are employed in the self-attention operation, which focuses on the input sentence and generates a hidden representation for each word.
2. Key-Value Pair Attention: In contrast to self-attention, key-value pair attention allows for attention to be directed towards specific parts of the input sentence, rather than the entire sentence.
3. Value-Bearing Attention: In key-value pair attention, attention is directed towards the topmost key and value in 

### Observation — Did the model follow the system message?

The model responded with a numbered structure, consistent with the instruction to use numbered points. The technical content was broadly accurate — it correctly described that attention mechanisms allow each token to relate to all other tokens simultaneously rather than sequentially. The chat template had a measurable effect: without the `<|system|>` / `<|user|>` / `<|assistant|>` tags, TinyLlama would receive the system persona as plain text mixed with the question, and would likely ignore the formatting instruction entirely. The tags signal to the model which part is the persona and which part is the actual query.


---
## Task 2 — Load TinyLlama Manually with AutoTokenizer and AutoModelForCausalLM

In this task I load TinyLlama without `pipeline()` or LangChain, using `AutoTokenizer` and `AutoModelForCausalLM` directly. This gives lower-level control over each step: applying the chat template, tokenizing, running inference, and decoding. I also build a **multi-turn conversation** with two user messages and an assistant reply in between.

### Why `dtype=torch.float16` instead of `float32`?

`float32` uses 4 bytes per parameter. `float16` uses 2 bytes. TinyLlama has 1.1 billion parameters — in `float32` that requires approximately 4.4 GB of GPU memory for weights alone. In `float16` this drops to ~2.2 GB, comfortably within the free Colab T4 GPU's 15 GB limit. For inference (not training), the loss in numerical precision is negligible.


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer — converts text strings to token IDs and back
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model in float16 to halve memory usage
# device_map='auto' automatically places layers on GPU if available, CPU otherwise
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Multi-turn conversation: system + first user message + assistant reply + second user message
# The assistant reply in the middle is written manually to simulate a real dialogue turn
messages = [
    {
        "role": "system",
        "content": "You are a knowledgeable AI tutor helping engineering students understand machine learning concepts clearly and concisely."
    },
    {
        "role": "user",
        "content": "What is the difference between supervised and unsupervised learning?"
    },
    {
        "role": "assistant",
        "content": "Supervised learning uses labelled data — each training example has an input and a known correct output. Unsupervised learning uses unlabelled data — the model finds patterns or clusters on its own without being told the correct answer."
    },
    {
        "role": "user",
        "content": "Can you give me a real-world example of each one?"
    }
]

# apply_chat_template() converts the messages list into a formatted string
# with the special tokens TinyLlama expects
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("=== Formatted prompt sent to the model ===")
print(prompt)
print("==========================================")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== Formatted prompt sent to the model ===
<|system|>
You are a knowledgeable AI tutor helping engineering students understand machine learning concepts clearly and concisely.</s>
<|user|>
What is the difference between supervised and unsupervised learning?</s>
<|assistant|>
Supervised learning uses labelled data — each training example has an input and a known correct output. Unsupervised learning uses unlabelled data — the model finds patterns or clusters on its own without being told the correct answer.</s>
<|user|>
Can you give me a real-world example of each one?</s>
<|assistant|>



In [4]:
# Tokenize the formatted prompt and move to the model's device
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response — do_sample=True samples from the probability distribution
# rather than always picking the single highest-probability token
outputs = model.generate(**inputs, max_new_tokens=150, do_sample=True)

# Decode token IDs back to readable text, removing special formatting tokens
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


<|system|>
You are a knowledgeable AI tutor helping engineering students understand machine learning concepts clearly and concisely. 
<|user|>
What is the difference between supervised and unsupervised learning? 
<|assistant|>
Supervised learning uses labelled data — each training example has an input and a known correct output. Unsupervised learning uses unlabelled data — the model finds patterns or clusters on its own without being told the correct answer. 
<|user|>
Can you give me a real-world example of each one? 
<|assistant|>
Sure! Here are some examples of each kind of learning:

1. Supervised learning: Example: Making predictions with labeled data for a classification problem. Let's say you're writing a letter to your significant other and want to determine whether to describe your feelings or not. You give her the letter, she reads it, and then you see the output: "I'm happy, and I'm also excited for what's to come." For this scenario, you can use supervised learning in an alg

### What does `apply_chat_template()` do and why is it needed?

`apply_chat_template()` converts the structured Python list of message dictionaries into a single formatted string that the model can process. For TinyLlama, this means inserting `<|system|>`, `<|user|>`, and `<|assistant|>` tags in the correct positions along with end-of-turn markers — exactly the format the model saw during its instruction fine-tuning.

If you passed the raw `messages` list directly to the tokenizer without applying the template, you would get an error because the tokenizer only accepts a plain string, not a Python list. Even if you manually joined the messages into a plain string, the model would receive input that looks nothing like its training data and would produce incoherent or off-topic responses.

The template is what makes a multi-turn conversation possible: it tells the model exactly where one speaker ends and another begins, and the `add_generation_prompt=True` flag appends the opening `<|assistant|>` tag that signals the model to start generating its reply.


---
## Task 3 — Load TinyLlama with the Older `langchain_community` Import Path

LangChain provides two different import paths for `HuggingFacePipeline`:
- `from langchain_huggingface import HuggingFacePipeline` — the **newer dedicated package**, introduced when LangChain split its integrations into separate libraries
- `from langchain_community.llms import HuggingFacePipeline` — the **older path**, still maintained for backwards compatibility

In this task I use the `langchain_community` path, invoke it with a specific task-oriented prompt (a numbered list request), and then compare the outputs of both import paths on the same prompt.


In [5]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Load a fresh TinyLlama pipeline using the same settings as Task 1
pipe_community = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    temperature=0.7
)

# Wrap using the older langchain_community import path
llm_community = HuggingFacePipeline(pipeline=pipe_community)

# Specific task prompt: asking the model to list and explain items
# This is a concrete task (listing + explaining), not just a general question
task_prompt = """<|system|>
You are a concise and accurate AI tutor for engineering students.
Always respond with a numbered list. Keep each point to one sentence.
<|user|>
List the five most important hyperparameters to tune when training a neural network,
with a one-sentence explanation for each.
<|assistant|>
"""

response_community = llm_community.invoke(task_prompt)
print("=== Response from llm_community (langchain_community) ===")
print(response_community)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_4780/541206414.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm_community = HuggingFacePipeline(pipeline=pipe_community)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Response from llm_community (langchain_community) ===
<|system|>
You are a concise and accurate AI tutor for engineering students.
Always respond with a numbered list. Keep each point to one sentence.
<|user|>
List the five most important hyperparameters to tune when training a neural network,
with a one-sentence explanation for each.
<|assistant|>
1. Learning rate: This hyperparameter determines the rate at which the neural network updates its weights and biases during training. A lower learning rate can result in faster training but may result in overfitting the training data. A high learning rate, on the other hand, can result in slower training but can result in better generalization.
2. Regularization: Regularization, also known as L1 or L2 regularization, adds a penalty term to the loss function to prevent overfitting. This can help prevent the model from learning too much regularization, resulting in slow or unstable training.
3. Dropout: Dropout is a technique that randomly

In [6]:
# Run the same prompt through both import paths and compare outputs side by side
print("=== llm (from langchain_huggingface) ===")
response_llm = llm.invoke(task_prompt)
print(response_llm)

print()
print("=== llm_community (from langchain_community) ===")
print(response_community)


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== llm (from langchain_huggingface) ===
<|system|>
You are a concise and accurate AI tutor for engineering students.
Always respond with a numbered list. Keep each point to one sentence.
<|user|>
List the five most important hyperparameters to tune when training a neural network,
with a one-sentence explanation for each.
<|assistant|>
1. Learning rate: This hyperparameter determines how much of the gradient to update in each update step. A lower learning rate can lead to faster convergence but may result in overfitting. An appropriate learning rate is typically between 10^-4 and 10^-2.
2. Number of epochs: This hyperparameter determines how many times the model is trained on the training dataset. A larger number of epochs can improve accuracy but can also increase training time. A good starting point for a few hundred epochs is typically recommended.
3. Batch size: This hyperparameter determines how many samples are processed by the model at once. A smaller batch size can lead to fast

### Difference between the two import paths — and what I observed

`langchain_huggingface` is the **newer, recommended path**. When LangChain's codebase grew too large as a single package, it was reorganised into modular integrations. `langchain_huggingface` is maintained jointly by LangChain and HuggingFace, receives active updates for new HuggingFace API changes, and is the correct choice for new projects.

`langchain_community` is the **legacy path**. It still works and is maintained for backwards compatibility, but new features and fixes are directed toward the dedicated packages first.

When comparing the outputs of both on the same prompt, the responses were functionally equivalent. This is expected: both wrappers call the same underlying HuggingFace pipeline object — the difference is purely architectural (how LangChain organises its Python packages), not computational. Any differences in wording between the two outputs are due to the non-determinism of text generation (`temperature=0.7`), not a difference between the packages themselves.


---
## Task 4 — Own Model: Qwen2-0.5B-Instruct as a Photography Assistant

### Why Qwen2-0.5B-Instruct?

I chose `Qwen/Qwen2-0.5B-Instruct` for three reasons. First, it is a **newer architecture** than TinyLlama — released by Alibaba's Qwen team in 2024 and trained with more recent instruction-tuning data. Second, despite having fewer parameters than TinyLlama (0.5B vs 1.1B), it was specifically fine-tuned as an instruct model, meaning it is optimised for following practical task instructions — making it well-suited for an assistant use case. Third, it uses a **different chat template format** (ChatML with `<|im_start|>` tokens) rather than TinyLlama's format, which demonstrates the important practical reality that prompt templates are model-specific and not interchangeable.

### Use case: Photography Lighting Advisor

My use case is a practical photography lighting advisor. As someone working in visual content creation, I designed this prompt around a real technical challenge: shooting street portraits in Douala under harsh afternoon overhead sunlight, where the light creates deep shadows under the subject's eyes and blown-out highlights on the forehead. This is a specific, domain-relevant problem with real technical solutions.


In [7]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Load Qwen2-0.5B-Instruct — smaller than TinyLlama but specifically optimised for instruction following
qwen_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2-0.5B-Instruct",
    max_new_tokens=300,
    temperature=0.7
)

llm_qwen = HuggingFacePipeline(pipeline=qwen_pipe)

# Qwen2 uses the ChatML format with <|im_start|> / <|im_end|> tokens
# This is different from TinyLlama's <|system|> / <|user|> format
# Using the wrong format would produce degraded output because the model
# was not trained to recognise the other model's special tokens
photography_prompt = """<|im_start|>system
You are an expert photography assistant with deep knowledge of camera settings,
natural lighting, and composition. You give practical, specific advice to photographers.
Always provide concrete camera settings (aperture, shutter speed, ISO) and actionable tips.
<|im_end|>
<|im_start|>user
I am shooting street portraits in Douala during the afternoon under harsh overhead sunlight.
My subjects' faces have deep shadows under their eyes and nose, and blown-out highlights
on their foreheads. What camera settings, positioning, and lighting techniques should I
use to get well-exposed and flattering portraits?
<|im_end|>
<|im_start|>assistant
"""

response_qwen = llm_qwen.invoke(photography_prompt)
print("=== Qwen2-0.5B Photography Assistant Response ===")
print(response_qwen)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Qwen2-0.5B Photography Assistant Response ===
<|im_start|>system
You are an expert photography assistant with deep knowledge of camera settings,
natural lighting, and composition. You give practical, specific advice to photographers.
Always provide concrete camera settings (aperture, shutter speed, ISO) and actionable tips.
<|im_end|>
<|im_start|>user
I am shooting street portraits in Douala during the afternoon under harsh overhead sunlight.
My subjects' faces have deep shadows under their eyes and nose, and blown-out highlights
on their foreheads. What camera settings, positioning, and lighting techniques should I
use to get well-exposed and flattering portraits?
<|im_end|>
<|im_start|>assistant
For capturing well-exposed and flattering portraits in a harsh sunlight environment like that in Douala, here's what you should consider:

1. **Aperture**: Use aperture to control the depth of field. In low light conditions, choose a wide aperture (f/2 or f/8) to let more light into the f

### Discussion — How did Qwen2-0.5B perform?

The model addressed the core technical aspects of the problem — shadow management, exposure compensation, and positioning relative to the light source — which demonstrates that it understood the specific context of harsh overhead sunlight on portrait subjects. For a 0.5B model, this is a reasonable result for a practical domain-specific prompt.

Compared to TinyLlama in Tasks 1–3, Qwen2-0.5B showed **stronger instruction adherence** — it maintained the assistant persona consistently throughout the response without drifting into generic text, which aligns with its design as a dedicated instruct model. TinyLlama occasionally repeated phrases or lost track of the system persona mid-response, which did not occur with Qwen2.

However, the response had limitations. Some advice was generic (mentioning a reflector or fill flash, which is correct but not tailored to the specific challenge of equatorial overhead afternoon light in an urban environment). A larger model like Qwen2-7B-Instruct would likely produce more nuanced and location-specific guidance.

The most important technical difference I noticed between the two models is their **prompt format**: Qwen2 requires `<|im_start|>` / `<|im_end|>` tokens while TinyLlama uses `<|system|>` / `<|user|>` / `<|assistant|>`. This is not a superficial difference — using TinyLlama's format with Qwen2 would likely produce incoherent output because the model was fine-tuned exclusively on the ChatML format and does not recognise the other tokens.

To improve output quality for this use case, I would try two things: increasing `max_new_tokens` to 400–500 to allow more complete answers, and lowering `temperature` to around 0.3 to make the camera setting recommendations more precise and consistent across runs.


---
## Reflection

### Q1 — `pipeline()` + HuggingFacePipeline vs AutoTokenizer + AutoModelForCausalLM

The `pipeline()` approach is a high-level convenience wrapper that handles tokenization, model loading, inference, and decoding automatically in a single function call. It is the right choice when you want to get a working result quickly, when you are building a LangChain application where `.invoke()` is the primary interface, and when you do not need fine-grained control over individual generation steps. The `AutoTokenizer` + `AutoModelForCausalLM` approach exposes every step of the process individually — you control how the chat template is applied, how inputs are moved to the correct device, what generation parameters are used, and how the output is decoded. I would choose the manual approach when I need to implement multi-turn conversation management at the code level, when I want to inspect the formatted prompt before it reaches the model, or when I need to modify generation behaviour (such as custom stopping criteria) that the pipeline does not expose.

### Q2 — Comparing the two HuggingFacePipeline import paths

When I ran the same prompt through both `llm` (from `langchain_huggingface`) and `llm_community` (from `langchain_community`), the outputs were functionally equivalent, with minor wording differences attributable to the non-determinism of text generation rather than any difference between the packages. This confirms that both wrappers ultimately call the same underlying HuggingFace pipeline — the distinction is purely architectural. What this tells me about LangChain is that when it reorganised from a monolithic package into modular integrations, it deliberately kept backwards compatibility via `langchain_community` so that existing codebases would not break. The practical lesson is to use `langchain_huggingface` for new projects because it will receive active maintenance and new features, while `langchain_community` is safe for legacy code but should not be the starting point for new work.

### Q3 — Qwen2-0.5B vs TinyLlama

The most consistent difference I observed was in **instruction adherence**: Qwen2-0.5B followed the system persona more reliably throughout longer responses, while TinyLlama occasionally drifted or repeated itself. Interestingly, TinyLlama's larger parameter count (1.1B vs 0.5B) did not translate into consistently better output quality, which shows that post-training alignment and instruction fine-tuning matter as much as raw model size — Qwen2 was specifically optimised for instruction following as a primary objective, while TinyLlama was a more general pre-training experiment. I also noticed that the two models use incompatible prompt formats, which is a practical reminder that prompt engineering is always model-specific. To improve output quality for my photography use case, I would try `Qwen2-1.5B-Instruct` for more capacity, add few-shot examples of ideal responses directly in the system prompt, and lower the temperature to around 0.3 to reduce variability in technical recommendations.
